In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import csv
import numpy as np
import geopandas as gpd
import rasterio

from PIL import Image
from tqdm.auto import tqdm
from rasterio.windows import from_bounds, Window

print("Libraries imported successfully.")

Mounted at /content/drive
Libraries imported successfully.


In [2]:
ROOT = "/content/drive/MyDrive/RoofMaterialProject"

REGIONS = {
    "borde_rural": {
        "image": f"{ROOT}/borde_rural/borde_rural_ortho-cog.tif",
        "geojson": f"{ROOT}/borde_rural/test-borde_rural.geojson"
    },

    "borde_soacha": {
        "image": f"{ROOT}/borde_soacha/borde_soacha_ortho-cog.tif",
        "geojson": f"{ROOT}/borde_soacha/test-borde_soacha.geojson"
    }
}

# Separate test output folder.
# This will NOT modify your existing training patches.

OUTPUT = f"{ROOT}/patches/test"

os.makedirs(OUTPUT, exist_ok=True)

MANIFEST = f"{OUTPUT}/test_patch_manifest.csv"

PATCH_SIZE = 128
CONTEXT = 0.15

print("Output folder:", OUTPUT)
print("Patch size:", PATCH_SIZE)
print("Context margin:", CONTEXT)

Output folder: /content/drive/MyDrive/RoofMaterialProject/patches/test
Patch size: 128
Context margin: 0.15


In [3]:
def extract_patch(src, geometry, output_path):

    # Get building boundary coordinates
    minx, miny, maxx, maxy = geometry.bounds

    # Add 15% surrounding context on each side
    pad = max(maxx - minx, maxy - miny) * CONTEXT

    minx -= pad
    miny -= pad
    maxx += pad
    maxy += pad

    # Convert map coordinates into a pixel window
    window = from_bounds(
        minx, miny, maxx, maxy,
        transform=src.transform
    )

    # Round window dimensions
    window = window.round_offsets().round_lengths()

    # Keep window inside the TIFF image
    full_window = Window(0, 0, src.width, src.height)
    window = window.intersection(full_window)

    if window.width < 2 or window.height < 2:
        raise ValueError("Crop is too small or outside the image")

    # Read RGB bands only
    rgb = src.read(
        [1, 2, 3],
        window=window,
        boundless=False
    )

    if rgb.size == 0:
        raise ValueError("Empty image crop")

    # Convert bands, height, width to height, width, bands
    rgb = np.moveaxis(rgb, 0, -1)

    # Convert to RGB image
    image = Image.fromarray(
        rgb.astype(np.uint8),
        mode="RGB"
    )

    # Resize to 128 × 128
    image = image.resize(
        (PATCH_SIZE, PATCH_SIZE),
        Image.Resampling.BILINEAR
    )

    # Save image
    image.save(output_path, quality=95)

    return True

In [4]:
for region, paths in REGIONS.items():

    print("\n================================")
    print("Region:", region)
    print("================================")

    print("Image exists:", os.path.exists(paths["image"]))
    print("GeoJSON exists:", os.path.exists(paths["geojson"]))

    if not os.path.exists(paths["geojson"]):
        continue

    buildings = gpd.read_file(paths["geojson"])

    print("Total buildings:", len(buildings))
    print("Columns:", buildings.columns.tolist())
    print("ID column available:", "id" in buildings.columns)

    if "id" not in buildings.columns:
        print("ERROR: Building ID column missing.")

    if "roof_material" in buildings.columns:
        print("Note: Roof labels are present, but will NOT be used.")
    else:
        print("Unlabelled test GeoJSON confirmed.")


Region: borde_rural
Image exists: True
GeoJSON exists: True
Total buildings: 2160
Columns: ['id', 'geometry']
ID column available: True
Unlabelled test GeoJSON confirmed.

Region: borde_soacha
Image exists: True
GeoJSON exists: True
Total buildings: 3017
Columns: ['id', 'geometry']
ID column available: True
Unlabelled test GeoJSON confirmed.


In [5]:
# --------------------------------------------------
# 1. Manifest configuration
# --------------------------------------------------

fieldnames = [
    "region",
    "building_id",
    "patch_path",
    "status",
    "error"
]

# Load previous extraction progress if available
if os.path.exists(MANIFEST):

    with open(MANIFEST, "r", newline="", encoding="utf-8") as f:
        old_rows = list(csv.DictReader(f))

else:
    old_rows = []


# Keep successfully extracted patches
completed = {
    (r["region"], r["building_id"])
    for r in old_rows
    if r["status"] == "success"
    and os.path.exists(r["patch_path"])
}

# Store records using unique region + building ID
rows_by_key = {
    (r["region"], r["building_id"]): r
    for r in old_rows
}


# --------------------------------------------------
# 2. Process both geographical areas
# --------------------------------------------------

for region, paths in REGIONS.items():

    print(f"\nProcessing region: {region}")

    # Check input files
    if not os.path.exists(paths["image"]):
        print("IMAGE FILE NOT FOUND:", paths["image"])
        continue

    if not os.path.exists(paths["geojson"]):
        print("GEOJSON FILE NOT FOUND:", paths["geojson"])
        continue

    # Load unlabelled building polygons
    buildings = gpd.read_file(paths["geojson"])

    if "id" not in buildings.columns:
        print("Building ID column missing:", region)
        continue

    # Create region-specific output folder
    region_folder = os.path.join(OUTPUT, region)
    os.makedirs(region_folder, exist_ok=True)

    with rasterio.open(paths["image"]) as src:

        print("Image CRS:", src.crs)
        print("GeoJSON CRS:", buildings.crs)
        print("Total buildings:", len(buildings))

        # Reproject polygons to aerial image CRS
        buildings = buildings.to_crs(src.crs)

        for _, building in tqdm(
            buildings.iterrows(),
            total=len(buildings),
            desc=region
        ):

            building_id = str(building["id"])
            key = (region, building_id)

            # Skip already completed patches
            if key in completed:
                continue

            patch_path = os.path.join(
                region_folder,
                f"{building_id}.jpg"
            )

            try:

                # Validate geometry
                if building.geometry is None:
                    raise ValueError("Missing geometry")

                if building.geometry.is_empty:
                    raise ValueError("Empty geometry")

                if not building.geometry.is_valid:
                    raise ValueError("Invalid polygon geometry")

                # Extract and save patch
                extract_patch(
                    src,
                    building.geometry,
                    patch_path
                )

                record = {
                    "region": region,
                    "building_id": building_id,
                    "patch_path": patch_path,
                    "status": "success",
                    "error": ""
                }

                completed.add(key)

            except Exception as e:

                record = {
                    "region": region,
                    "building_id": building_id,
                    "patch_path": patch_path,
                    "status": "failed",
                    "error": str(e)
                }

            rows_by_key[key] = record

            # Save progress after every building
            with open(
                MANIFEST,
                "w",
                newline="",
                encoding="utf-8"
            ) as f:

                writer = csv.DictWriter(
                    f,
                    fieldnames=fieldnames
                )

                writer.writeheader()
                writer.writerows(rows_by_key.values())

    print(
        f"{region} completed. "
        f"Successful patches so far: {len(completed)}"
    )


# --------------------------------------------------
# 3. Final extraction report
# --------------------------------------------------

all_rows = list(rows_by_key.values())

success_rows = [
    r for r in all_rows
    if r["status"] == "success"
]

failed_rows = [
    r for r in all_rows
    if r["status"] == "failed"
]

print("\n========== TEST EXTRACTION REPORT ==========")

print("Successful patches:", len(success_rows))
print("Failed patches:", len(failed_rows))
print("Manifest:", MANIFEST)

for region in REGIONS:

    count = sum(
        r["region"] == region
        and r["status"] == "success"
        for r in all_rows
    )

    print(f"{region}: {count} successful patches")

if failed_rows:

    print("\nFirst 10 failures:")

    for r in failed_rows[:10]:
        print(
            r["region"],
            r["building_id"],
            r["error"]
        )


Processing region: borde_rural
Image CRS: EPSG:32618
GeoJSON CRS: EPSG:4326
Total buildings: 2160


borde_rural:   0%|          | 0/2160 [00:00<?, ?it/s]

/tmp/ipykernel_1124/4004503432.py:44: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


borde_rural completed. Successful patches so far: 2160

Processing region: borde_soacha
Image CRS: EPSG:32618
GeoJSON CRS: EPSG:4326
Total buildings: 3017


borde_soacha:   0%|          | 0/3017 [00:00<?, ?it/s]

/tmp/ipykernel_1124/4004503432.py:44: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


borde_soacha completed. Successful patches so far: 5177

========== TEST EXTRACTION REPORT ==========
Successful patches: 5177
Failed patches: 0
Manifest: /content/drive/MyDrive/RoofMaterialProject/patches/test/test_patch_manifest.csv
borde_rural: 2160 successful patches
borde_soacha: 3017 successful patches


In [6]:
import pandas as pd

df = pd.read_csv(MANIFEST)

print("Total buildings processed:", len(df))
print("Successfully extracted:", (df["status"] == "success").sum())
print("Failed extractions:", (df["status"] == "failed").sum())

print("\nRegion-wise results:")
print(df.groupby(["region", "status"]).size())

Total buildings processed: 5177
Successfully extracted: 5177
Failed extractions: 0

Region-wise results:
region        status 
borde_rural   success    2160
borde_soacha  success    3017
dtype: int64
